In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, regexp_replace, round

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("LendingClub_Silver_Cleaning") \
    .getOrCreate()

# 2. Read the Bronze Data
# Based on your screenshot, the file is inside a subfolder with the same name
csv_path = "../data/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv"

print("Starting to read the 1GB file... this might take 2-4 minutes.")
df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)

print(f"Raw Data Count: {df_raw.count()}")

Starting to read the 1GB file... this might take 2-4 minutes.
Raw Data Count: 2260701


In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, regexp_replace, round

# --- FORCE WINDOWS HADOOP PATHS (Fixes the absolute path error) ---
os.environ['HADOOP_HOME'] = r'D:\hadoop'
os.environ['hadoop.home.dir'] = r'D:\hadoop'   # Key: tell JVM directly
os.environ['PATH'] = r'D:\hadoop\bin;' + os.environ['PATH']
# ------------------------------------------------------------------

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("LendingClub_Silver_Cleaning") \
    .getOrCreate()

# 2. Read the Bronze Data
csv_path = "../data/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv"

print("Starting to read the 1GB file... this might take 2-4 minutes.")
df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)

print(f"Raw Data Count: {df_raw.count()}")

# ---------------------------------------------------------
# (If splitting into cells, start a new cell here)
# ---------------------------------------------------------

# 3. Clean 'int_rate' (e.g., "13.5%" -> 13.5)
df_clean = df_raw.withColumn("int_rate", 
    round(regexp_replace(col("int_rate"), "%", "").cast("float"), 2)
)

# 4. Clean 'term' (e.g., " 36 months" -> 36)
df_clean = df_clean.withColumn("term", 
    regexp_replace(col("term"), " months", "").cast("integer")
)

# 5. Drop Leakage Columns
leakage_cols = [
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int", 
    "total_rec_late_fee", "recoveries", "collection_recovery_fee", 
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "out_prncp", "out_prncp_inv"
]
df_clean = df_clean.drop(*leakage_cols)

# 6. Handle Missing Target
valid_statuses = ["Fully Paid", "Charged Off", "Default"]
df_clean = df_clean.filter(col("loan_status").isin(valid_statuses))

# 7. Create the ML Target Variable (1 for Default, 0 for Fully Paid)
df_clean = df_clean.withColumn("is_default", 
    when(col("loan_status").isin(["Charged Off", "Default"]), 1).otherwise(0)
)

print(f"Cleaned Data Count: {df_clean.count()}")
print("Nulls in int_rate:", df_clean.filter(col("int_rate").isNull()).count())

# ---------------------------------------------------------
# (If splitting into cells, start a new cell here)
# ---------------------------------------------------------

# 8. SAVE THE SILVER DATA
print("Saving Silver data to Parquet... (Please wait)")
df_clean.write.mode("overwrite").parquet("../data/silver_lending_club_cleaned.parquet")
print("Silver layer saved successfully!")

Starting to read the 1GB file... this might take 2-4 minutes.
Raw Data Count: 2260701
Cleaned Data Count: 1345349
Nulls in int_rate: 0
Saving Silver data to Parquet... (Please wait)
Silver layer saved successfully!


In [1]:
df_clean.write.mode("overwrite").parquet("../data/silver_lending_club_cleaned.parquet")
print("Silver layer saved successfully!") 

NameError: name 'df_clean' is not defined